In [ ]:
!pip install mordred
!pip install rdkit-pypi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.8/128.8 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.5 MB/s eta 0:00:00
  Created wheel for mordred: filename=mordred-1.2.0-py3-none-any.whl size=176718 sha256=4dadaa7c390389481e232e8dd79b0b43a4789fe06da16bb21a069d7e2d9322ca
  Stored in directory: /root/.cache/pip/wheels/a7/4f/b8/d4c6591f6ac944aaced7865b349477695f662388ad958743c7
Successfully built mordred
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nx-cugraph-cu12 24.10.0 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 55.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from multiprocessing import freeze_support
from rdkit import Chem
from mordred import Calculator, descriptors, get_descriptors_in_module
from mordred import VertexAdjacencyInformation, ExtendedTopochemicalAtom, AdjacencyMatrix,BCUT,DetourMatrix,Autocorrelation, BaryszMatrix, CPSA, Chi, EState, InformationContent, KappaShapeIndex, MoRSE, MolecularDistanceEdge,ZagrebIndex

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving input_smiles.csv to input_smiles.csv


In [ ]:
# Check the None type in the mol list, return the indices
# remove the complex entries
def count_none_elements(input_list):
    none_count = 0
    none_indices = []

    for index, item in enumerate(input_list):
        if item is None:
            none_count += 1
            none_indices.append(index)

    return none_count, none_indices

def remove_entry(df,none_indices):
    ref_set = set([df.at[row,'refcode'] for row in none_indices])
    for ref in ref_set:
        df = df.drop(df[df['refcode']==ref].index)
    return df

In [ ]:
def load_lig(file,drop_flag=False,*drop_ref:str):
# load ligand database
    df_ligand = pd.read_csv("input_smiles.csv",index_col=0)
    if drop_flag==True:
        print('dropping assigned rows')
        print('original df rows:',df_ligand.shape[0])
        print('removed rows:',df_ligand[df_ligand['refcode'].isin(drop_ref)].index)
        df_ligand = df_ligand.drop(df_ligand[df_ligand['refcode'].isin(drop_ref)].index)
        print('new df rows:',df_ligand.shape[0])

    mols = [Chem.MolFromSmiles(smi) for smi in df_ligand.loc[:,'SMILES']]
    none_result = count_none_elements(mols)
    if none_result[0]!=0:
        print('none count:',none_result[0])
        print('none indices:',none_result[1])
        print('ref:',[df_ligand.at[row,'refcode'] for row in none_result[1]])
        print('original df rows before removing none:',df_ligand.shape[0])
        df_ligand = remove_entry(df_ligand,none_result[1])
        print('new df rows after removing none objects:',df_ligand.shape[0])
#         need to remove the same entries in df_metal too

    df_ligand = df_ligand.reset_index(drop=True)
    return df_ligand

In [ ]:
def cal_desc(df_ligand,out_path):
    descs = get_descriptors_in_module(descriptors,submodule=True)

#     exclude certain descriptors
    exclude_list = ['mZagreb1','mZagreb2','HybRatio','TopoShapeIndex','PetitjeanIndex','VR3_D','Vabc']
    descs = filter(lambda d:  ((d.__module__ != Autocorrelation.__name__) and
                          (d.__module__ != DetourMatrix.__name__) and
                          (d.__module__ != BaryszMatrix.__name__) and
                          (d.__module__ != CPSA.__name__) and
                          (d.__module__ != EState.__name__) and
                          (d.__module__ != MoRSE.__name__) and
                          (d.__module__ != BCUT.__name__) and
                          (d.__module__ != Chi.__name__) and
                          (d.__module__ != InformationContent.__name__) and
                          (d.__module__ != AdjacencyMatrix.__name__) and
                          (d.__module__ != KappaShapeIndex.__name__) and
                           (d.__module__ != ExtendedTopochemicalAtom.__name__) and
                           (d.__module__ != VertexAdjacencyInformation.__name__) and
#                           (getattr(d,'__name__') not in exclude_list) and
                          (d.__module__ != MolecularDistanceEdge.__name__)), descs)

#     create mol objects from SMILES
    mols = [Chem.MolFromSmiles(smi) for smi in df_ligand.loc[:,'SMILES']]
# Check the None type in the mol list, return the indices
    none_result = count_none_elements(mols)
    if none_result[0]!=0:
        print('none count:',none_result[0])
        print('none indices',none_result[1])
    else:
        freeze_support()
        calc = Calculator(descs, ignore_3D = True)
        result = calc.pandas(mols)

        #     exclude certain descriptors
        for ex in exclude_list:
            if ex in result.columns:
                del result[ex]

        # replace non-number items in RotRatio with 0
        result['RotRatio'] = pd.to_numeric(result['RotRatio'], errors='coerce').fillna(0)
    #     convert T/F to (1,0)
        result['Lipinski']= result['Lipinski'].astype(int)
        result['GhoseFilter']= result['GhoseFilter'].astype(int)

    # check if all items are numbers
        numeric_df = result.select_dtypes(include=[np.number])
        all_numeric = numeric_df.shape[1] == result.shape[1]
        if all_numeric == False:
            all_columns = set(result.columns)
            numeric_columns = set(numeric_df.columns)
            non_numeric_columns = all_columns - numeric_columns
            print("Columns that contain non-numeric items:", non_numeric_columns)
        if df_ligand.shape[0]==result.shape[0]:
            result = pd.concat([df_ligand,result],axis=1)
        else:
            print('rows of df_ligand:',df_ligand.shape[0])
            print('rows of result:',result.shape[0])
        result.to_csv(out_path)


In [ ]:
df_Cr_ligand = load_lig('input_smiles.csv')


In [ ]:
df_desc_Cr_ligand = cal_desc(df_Cr_ligand,'Cr_ligand_desc.csv')

100%|██████████| 1186/1186 [00:50<00:00, 23.56it/s]


Columns that contain non-numeric items: {'ABCGG', 'ABC'}


In [ ]:
files.download('Cr_ligand_desc.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>